# MAGIC_UNET Replication

## Import libraries

In [1]:
import numpy as np
import sys

sys.path.insert(1, 'CDIP/')
from CDIPutils import accuracy, SQUIDify, magnitude, batchMagnitude

import torch
from TorchUnet import UNet, HPOUNet
import tensorflow as tf

import FourierUtils
from importlib import reload

2026-08-21 15:47:00.501797: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-21 15:47:00.569712: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-21 15:47:01.773722: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Load model & data and perform inference

### 64x64 50um Out of Distribution

In [139]:
class config(object):
    im_h = 64
    in_im_c = 1
    out_im_c = 2
    avgPool = True

    seed = 42
    layers = [64, 128, 256, 512]

model = HPOUNet(config)
model.load_state_dict(torch.load("models/CDIP_SQUID_B_noise/model.pt", weights_only=True))
print(model.eval())

HPOUNet(
  (inc): DoubleConv(
    (double_conv): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (downLayers): ModuleList(
    (0): Down(
      (maxpool_conv): Sequential(
        (0): AvgPool2d(kernel_size=2, stride=2, padding=0)
        (1): DoubleConv(
          (double_conv): Sequential(
            (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
            (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1

In [149]:
inputData = np.load("../MAGIC_UNET/Data/OutOfDistribution/OutOfDistVal_64x64_50um_Bxyz.npy")
labelData = np.load("../MAGIC_UNET/Data/OutOfDistribution/OutOfDistVal_64x64_50um_Jxy.npy")

inputData = inputData[:, :, :, 0:3]
# print(inputData.shape)

inputData = inputData.swapaxes(1,3).astype('float32')
labelData = labelData.astype('float32')
print(inputData.shape, labelData.shape)

inputData = SQUIDify(inputData, 62, 0)
# labelData = SQUIDify(labelData, 62, 0)

normInData = (inputData - inputData.mean()) / inputData.std()
normLabelData = (labelData - labelData.mean()) / labelData.std()

# normInDataTensor = torch.from_numpy(normInData.swapaxes(1,3).astype('float32'))
predData = model(normInData)
predData = predData.detach().numpy().swapaxes(1,3)

#predData = magnitude(predData)
#normLabelData = magnitude(normLabelData)

print(predData.shape, normLabelData.shape)

avSSIM, avRMSE, avPSNR = accuracy(np.squeeze(normLabelData),
                                  np.squeeze(predData),
                                  convMagnitude=True)
print("Average SSIM:", avSSIM)
print("Average RMSE:", avRMSE)
print("Average PSNR:", avPSNR)

(100, 3, 64, 64) (100, 64, 64, 2)
(100, 64, 64, 2) (100, 64, 64, 2)
Average SSIM: 0.7307413859020193
Average RMSE: 2.301070911760513
Average PSNR: 19.6961433804163


### 64x64 50um In-Distribution

In [ ]:
inputData = np.load("../MAGIC_UNET/Data/InDistributionVal/validation_64x64_50um_Bxyz.npy")
labelData = np.load("../MAGIC_UNET/Data/InDistributionVal/validation_64x64_50um_Jxy.npy")

inputData = inputData[:, :, :, 0:3]
# print(inputData.shape)

inputData = inputData.swapaxes(1,3).astype('float32')
labelData = labelData.astype('float32')

inputData = SQUIDify(inputData, 62, 0)
# labelData = SQUIDify(labelData, 62).numpy().swapaxes(1,3)

normInData = (inputData - inputData.mean()) / inputData.std()
normLabelData = (labelData - labelData.mean()) / labelData.std()

# normInDataTensor = torch.from_numpy(normInData.swapaxes(1,3).astype('float32'))
predData = model(normInData)
predData = predData.detach().numpy().swapaxes(1,3)

predData = magnitude(predData)
normLabelData = magnitude(normLabelData)

avSSIM, avRMSE, avPSNR = accuracy(np.squeeze(normLabelData), np.squeeze(predData), magnitude=False)
print("Average SSIM:", avSSIM)
print("Average RMSE:", avRMSE)
print("Average PSNR:", avPSNR)

Average SSIM: 0.6154218846489152
Average RMSE: 0.29409519386378796
Average PSNR: 21.75135100072243


### 64x64 500um In-Distribution

In [ ]:
inputData = np.load("../MAGIC_UNET/Data/InDistributionVal/validation_64x64_500um_Bxyz.npy")
labelData = np.load("../MAGIC_UNET/Data/InDistributionVal/validation_64x64_500um_Jxy.npy")

inputData = inputData[:, :, :, 0:3]
print(inputData.shape)

inputData = inputData.swapaxes(1,3).astype('float32')
labelData = labelData.astype('float32')

inputData = SQUIDify(inputData, 62, 0)
# labelData = SQUIDify(labelData, 62).numpy().swapaxes(1,3)

normInData = (inputData - inputData.mean()) / inputData.std()
normLabelData = (labelData - labelData.mean()) / labelData.std()

# normInDataTensor = torch.from_numpy(normInData.swapaxes(1,3).astype('float32'))
predData = model(normInData)
predData = predData.detach().numpy().swapaxes(1,3)

predData = magnitude(predData)
normLabelData = magnitude(normLabelData)

avSSIM, avRMSE, avPSNR = accuracy(np.squeeze(normLabelData), np.squeeze(predData), magnitude=False)
print("Average SSIM:", avSSIM)
print("Average RMSE:", avRMSE)
print("Average PSNR:", avPSNR)

(2048, 64, 64, 3)
Average SSIM: 0.045377169506252216
Average RMSE: 0.937240750537666
Average PSNR: -6.645613791488772


# MAGIC_UNet

In [ ]:
def UNET_normalized_inference(data_in_temp, model):
    
    # Calculate vector norms
    vect_norm = np.array(data_in_temp[:, :, :, 0]**2 +
                 data_in_temp[:, :, :, 1]**2 +
                 data_in_temp[:, :, :, 2]**2)**0.5
    vect_norm = np.max(vect_norm, axis=1)
    vect_norm = np.max(vect_norm, axis=1)

    file_size = np.size(vect_norm)
    for j in range(file_size):
        if vect_norm[j] != 0:
            data_in_temp[j, :, :, :] = data_in_temp[j, :, :, :] / vect_norm[j]

    # Make predictions
    predData = model(data_in_temp)["conv2d_101"]
    data_out_temp = np.zeros([file_size, 64, 64, 1])

    for j in range(file_size):
        if vect_norm[j] != 0:
            data_out_temp[j, :, :, 0] = predData[j, :, :, 0] * vect_norm[j] * 1e11
    return data_out_temp

In [ ]:
model = tf.keras.layers.TFSMLayer("../MAGIC_UNET/MAGIC_UNet_64x64_50um/checkpoint/", call_endpoint="serving_default")

I0000 00:00:1786977091.050814 3325041 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22322 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:3b:00.0, compute capability: 8.6
I0000 00:00:1786977091.052342 3325041 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22322 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:af:00.0, compute capability: 8.6


## 50um Out-of-distirbution

In [ ]:
inputData = np.load("../MAGIC_UNET/Data/OutOfDistribution/OutOfDistVal_64x64_50um_Bxyz.npy")
labelData = np.load("../MAGIC_UNET/Data/OutOfDistribution/OutOfDistVal_64x64_50um_Jxy.npy")

inputData = inputData[:, :, :, 0:3]

predData = UNET_normalized_inference(inputData, model)

normLabelData = magnitude(labelData)

avSSIM, avRMSE, avPSNR = accuracy(np.squeeze(normLabelData),
                                  np.squeeze(predData),
                                  magnitude=False)
print("Average SSIM:", avSSIM)
print("Average RMSE:", avRMSE)
print("Average PSNR:", avPSNR)

Average SSIM: 0.9240642429428664
Average RMSE: 7557319.507778672
Average PSNR: 18.54688507195023


# In-Distribution

In [ ]:
inputData = np.load("../MAGIC_UNET/Data/InDistributionVal/validation_64x64_500um_Bxyz.npy")
labelData = np.load("../MAGIC_UNET/Data/InDistributionVal/validation_64x64_500um_Jxy.npy")

inputData = inputData[:, :, :, 0:3]

predData = UNET_normalized_inference(inputData, model)

#normLabelData = magnitude(labelData)

avSSIM, avRMSE, avPSNR = accuracy(np.squeeze(labelData),
                                  np.squeeze(predData),
                                  magnitude=True)

print("Average SSIM:", avSSIM)
print("Average RMSE:", avRMSE)
print("Average PSNR:", avPSNR)

<class 'numpy.ndarray'>


TypeError: 'bool' object is not callable

# Fourier Analysis

In [12]:
inputData = np.load("../MAGIC_UNET/Data/OutOfDistribution/OutOfDistVal_64x64_50um_Bxyz.npy")
labelData = np.load("../MAGIC_UNET/Data/OutOfDistribution/OutOfDistVal_64x64_50um_Jxy.npy")

inputData = inputData[:, :, :, 0:3]

inputData = inputData.swapaxes(1,3).astype('float32')
labelData = labelData.astype('float32')

inputData = SQUIDify(inputData, 62, 0)
# labelData = SQUIDify(labelData, 62, 0)

inputData = inputData.swapaxes(1,3)

normInData = (inputData - inputData.mean()) / inputData.std()
normLabelData = (labelData - labelData.mean()) / labelData.std()

In [13]:
reload(FourierUtils)
fft = FourierUtils.FFT(normInData)

In [14]:
def Fourier(normInData):
    b_fourier_shifted=fft.dotheFFTsShift(normInData)
    b_fourier_shifted_blurred = fft.blurBs(b_fourier_shifted, fft.gauss_filt_strength)

    jx_fourier, jy_fourier = fft.fourierj_from_fourierb(b_fourier_shifted_blurred) # bx_fourier_shifted_blurred, by_fourier_shifted_blurred, bz_fourier_shifted_blurred)
    Jx_simulated, Jy_simulated = fft.dotheiFFTsJShift([jx_fourier, jy_fourier])

    data_out_fourier=np.zeros([fft.asize, fft.pixel_number_x, fft.pixel_number_x,2])
    data_out_fourier[:,:,:,0]=np.real(Jx_simulated)
    data_out_fourier[:,:,:,1]=np.real(Jy_simulated)
    # magPredData=np.sqrt(data_out_fourier[:,:,:,0]**2+data_out_fourier[:,:,:,1]**2)
    
    return data_out_fourier

print(normInData.shape)
predData = Fourier(normInData)

torch.Size([100, 64, 64, 1])


In [ ]:
print(predData.shape, normLabelData.shape)

predData = batchMagnitude(predData)
normLabelData = batchMagnitude(normLabelData)

avSSIM, avRMSE, avPSNR = accuracy(np.squeeze(normLabelData), np.squeeze(predData), convMagnitude=False)
print("Average SSIM:", avSSIM)
print("Average RMSE:", avRMSE)
print("Average PSNR:", avPSNR)

(100, 64, 64, 2) (100, 64, 64, 2)
(100, 64, 64, 2)
Average SSIM: 0.031087831169576374
Average RMSE: 116979423099.36235
Average PSNR: 11.786662584386214
